In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Read Bronze tables

orders = spark.table(
    "logistics.bronze.shipment_orders"
)

tracking = spark.table(
    "logistics.bronze.shipment_tracking_events"
)

attempts = spark.table(
    "logistics.bronze.delivery_attempts"
)

carriers = spark.table(
    "logistics.bronze.carrier_master"
)

warehouses = spark.table(
    "logistics.bronze.warehouse_master"
)

routes = spark.table(
    "logistics.bronze.route_master"
)

In [0]:
# Section 2 - Clean column names and string values

def clean_strings(df):
    # column names: trim, lowercase, spaces -> underscore
    df = df.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])
    # string values: trim, empty string -> null
    return df.withColumns({
        c: when(trim(col(c)) == "", None).otherwise(trim(col(c)))
        for c, t in df.dtypes if t == "string"
    })

orders     = clean_strings(orders)
tracking   = clean_strings(tracking)
attempts   = clean_strings(attempts)
carriers   = clean_strings(carriers)
warehouses = clean_strings(warehouses)
routes     = clean_strings(routes)

# Standardise case: codes -> UPPER, cities -> Initcap
orders = orders.withColumn("shipment_status", upper(col("shipment_status")))
orders = orders.withColumn("service_level", upper(col("service_level")))
orders = orders.withColumn("destination_city", initcap(col("destination_city")))

tracking = tracking.withColumn("event_type", upper(col("event_type")))
tracking = tracking.withColumn("event_city", initcap(col("event_city")))

attempts = attempts.withColumn("attempt_result", upper(col("attempt_result")))
attempts = attempts.withColumn("failure_reason", upper(col("failure_reason")))
attempts = attempts.withColumn("delivery_city", initcap(col("delivery_city")))

carriers = carriers.withColumn("active_flag", upper(col("active_flag")))
carriers = carriers.withColumn("service_region", initcap(col("service_region")))

warehouses = warehouses.withColumn("active_flag", upper(col("active_flag")))
warehouses = warehouses.withColumn("city", initcap(col("city")))

routes = routes.withColumn("active_flag", upper(col("active_flag")))
routes = routes.withColumn("destination_city", initcap(col("destination_city")))

display(orders.groupBy("shipment_status").count())

In [0]:
# Section 3 - Convert data types
# Failed casts are NOT treated as missing: raw Bronze value is kept and logged in cast_failures

from functools import reduce

TS_FMT = "yyyy-MM-dd'T'HH:mm:ss'Z'"
to_ts  = lambda c: try_to_timestamp(col(c), lit(TS_FMT))
to_dt  = lambda c: try_to_date(col(c), lit("yyyy-M-d"))
to_dbl = lambda c: col(c).try_cast("double")
to_int = lambda c: col(c).try_cast("int")
to_seq = lambda c: regexp_extract(col(c), r"(\d+)$", 1).try_cast("int")  # 'ATTEMP_4' -> 4

def convert_types(df, casts):
    """Keep original in <col>_raw, cast into <col>, list failed columns in _cast_errors."""
    df = df.withColumns({f"{c}_raw": col(c) for c in casts})
    df = df.withColumns({c: f(f"{c}_raw") for c, f in casts.items()})
    failed = [when(col(f"{c}_raw").isNotNull() & col(c).isNull(), lit(c)) for c in casts]
    return df.withColumn("_cast_errors", array_compact(array(*failed)))

def cast_failures(df, table, key, casts):
    return reduce(lambda a, b: a.unionByName(b), [
        df.filter(col(f"{c}_raw").isNotNull() & col(c).isNull())
          .select(lit(table).alias("table_name"), col(key).alias("record_id"),
                  lit(c).alias("column_name"), col(f"{c}_raw").alias("bronze_value"))
        for c in casts])

# table -> (dataframe, key, casts)
conversions = {
    "shipment_orders":          (orders,     "shipment_id",       {"created_ts": to_ts, "promised_delivery_ts": to_ts,
                                                                   "package_weight_kg": to_dbl, "shipping_cost": to_dbl}),
    "shipment_tracking_events": (tracking,   "tracking_event_id", {"event_ts": to_ts, "received_ts": to_ts}),
    "delivery_attempts":        (attempts,   "attempt_id",        {"attempt_ts": to_ts, "received_ts": to_ts, "attempt_sequence": to_seq}),
    "carrier_master":           (carriers,   "carrier_id",        {"onboarded_date": to_dt}),
    "warehouse_master":         (warehouses, "warehouse_id",      {"capacity_units": to_int, "opening_date": to_dt}),
    "route_master":             (routes,     "route_id",          {"distance_km": to_dbl, "planned_hours": to_dbl}),
}

typed, failures = {}, []
for table, (df, key, casts) in conversions.items():
    t = convert_types(df, casts)
    failures.append(cast_failures(t, table, key, casts))
    typed[table] = t.drop(*[f"{c}_raw" for c in casts])  # raw values live in cast_failures

orders, tracking, attempts = typed["shipment_orders"], typed["shipment_tracking_events"], typed["delivery_attempts"]
carriers, warehouses, routes = typed["carrier_master"], typed["warehouse_master"], typed["route_master"]

cast_failures_df = reduce(lambda a, b: a.unionByName(b), failures)

display(cast_failures_df.groupBy("table_name", "column_name").count().orderBy("table_name"))
display(cast_failures_df)

In [0]:
# Section 4 - Missing-value report (cast failures are counted separately, not as missing)

report_columns = {
    "shipment_orders": ["shipment_id", "order_id", "customer_id", "carrier_id", "origin_warehouse_id", "destination_city",
                        "created_ts", "promised_delivery_ts", "package_weight_kg", "service_level", "shipping_cost", "shipment_status"],
    "shipment_tracking_events": ["tracking_event_id", "shipment_id", "event_ts", "facility_id", "event_type", "event_city",
                                 "scan_source", "carrier_id", "scan_quality", "received_ts"],
    "delivery_attempts": ["attempt_id", "shipment_id", "attempt_ts", "driver_id", "attempt_result", "failure_reason",
                          "proof_of_delivery", "delivery_city", "attempt_sequence", "received_ts"],
    "carrier_master": ["carrier_id", "carrier_name", "service_region", "carrier_tier", "active_flag", "onboarded_date",
                       "contract_type", "service_level", "country", "support_group"],
    "warehouse_master": ["warehouse_id", "warehouse_name", "city", "region", "country", "capacity_units",
                         "warehouse_type", "active_flag", "opening_date", "manager_code"],
    "route_master": ["route_id", "origin_warehouse_id", "destination_city", "distance_km", "route_class",
                     "planned_hours", "service_region", "active_flag", "vehicle_class", "route_owner"],
}

rows = []
for table, cols in report_columns.items():
    df = typed[table]
    r = df.agg(
        count("*").alias("_total"),
        *[sum((col(c).isNull() & ~array_contains("_cast_errors", c)).cast("int")).alias(f"m_{c}") for c in cols],
        *[sum(array_contains("_cast_errors", c).cast("int")).alias(f"f_{c}") for c in cols],
    ).first()
    rows += [(table, c, r["_total"], r[f"m_{c}"], r[f"f_{c}"]) for c in cols]

missing_report = (spark.createDataFrame(rows, "dataset string, column_name string, total_rows long, missing_count long, cast_failure_count long")
                  .withColumn("missing_pct", round(col("missing_count") * 100 / col("total_rows"), 2)))

# Summary per dataset
display(missing_report.groupBy("dataset").agg(
    first("total_rows").alias("total_rows"),
    sum("missing_count").alias("missing_cells"),
    sum((col("missing_count") > 0).cast("int")).alias("columns_with_missing"),
    sum("cast_failure_count").alias("cast_failures")).orderBy("dataset"))

# Column-level detail (only columns with missing values or cast failures)
display(missing_report.filter("missing_count > 0 OR cast_failure_count > 0")
                      .orderBy("dataset", desc("missing_count")))

In [0]:
# Section 5 - Duplicate profiling
# Exact duplicate = same values in all business columns (lineage columns like _source_file ignored)
# Repeated key    = same business key appearing more than once (identical or conflicting rows)

business_keys = {
    "shipment_orders":          ["shipment_id"],
    "shipment_tracking_events": ["tracking_event_id"],
    "delivery_attempts":        ["attempt_id"],
    "carrier_master":           ["carrier_id"],
    "warehouse_master":         ["warehouse_id"],
    "route_master":            ["route_id"],
}

summary_rows, repeated_keys = [], {}
for table, keys in business_keys.items():
    df, cols = typed[table], report_columns[table]

    key_stats = (df.groupBy(*keys)
                   .agg(count("*").alias("occurrences"),
                        count_distinct(struct(*cols)).alias("distinct_versions"))
                   .filter("occurrences > 1"))
    repeated_keys[table] = key_stats

    r = df.agg(count("*").alias("total"), count_distinct(struct(*cols)).alias("distinct_rows")).first()
    k = key_stats.agg(count("*").alias("keys"),
                      coalesce(sum("occurrences"), lit(0)).alias("rows"),
                      coalesce(sum((col("distinct_versions") > 1).cast("int")), lit(0)).alias("conflicting")).first()

    summary_rows.append((table, ",".join(keys), r.total, r.distinct_rows, r.total - r.distinct_rows,
                         k.keys, k.rows, k.conflicting))

duplicate_report = spark.createDataFrame(summary_rows,
    "dataset string, business_key string, total_rows long, distinct_rows long, exact_duplicate_rows long, "
    "repeated_keys long, rows_with_repeated_key long, keys_with_conflicting_values long")
display(duplicate_report)

# Secondary business-event keys (same event recorded under different IDs)
display(spark.createDataFrame([
    ("shipment_tracking_events", "shipment_id,event_type,event_ts",
     tracking.groupBy("shipment_id", "event_type", "event_ts").count().filter("count > 1").count()),
    ("delivery_attempts", "shipment_id,attempt_sequence",
     attempts.groupBy("shipment_id", "attempt_sequence").count().filter("count > 1").count()),
], "dataset string, event_key string, repeated_keys long"))

# Detail: every copy of a repeated shipment_id with the file it came from
display(orders.join(repeated_keys["shipment_orders"], "shipment_id")
              .select("shipment_id", "occurrences", "distinct_versions", *report_columns["shipment_orders"][1:], "_source_file")
              .orderBy("shipment_id", "_source_file"))

In [0]:
# Section 6 - Write cleaned staging tables to silver
# All rows kept (duplicates are NOT removed); invalid values stay in place for later rules

silver_schema = "logistics.silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

for table in report_columns:
    clean_df = (typed[table]
                .withColumn("_silver_processed_ts", current_timestamp()))
    (clean_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{silver_schema}.clean_{table}"))
    print(f"Created {silver_schema}.clean_{table}")

# Verify: rows in vs rows out
display(spark.createDataFrame(
    [(t, typed[t].count(), spark.table(f"{silver_schema}.clean_{t}").count()) for t in report_columns],
    "dataset string, source_rows long, silver_rows long"))